# 03 AgentSHAP прогон для анализа в ноутбуке 02

**Нужно: локальная Ollama или вы можете аккуратно дотянуть с агентом платную API-шку.** 

```bash
ollama serve
ollama pull qwen2.5:3b
```

**Файлы по выполнению ноутбука:**
| Файл | Что содержит  |
|---|---|
| `finance.csv`, `finance_shapley.csv` | один `analyze()` на промпте с инструментами: коалиции, ответы, вызовы, значения Шепли |
| `noise_floor.json` | Потолок близости $\nu$ на трёх промптах с инструментами: сколько похожи два ответа на **одном и том же** тулсете |
| `notool_coalitions.csv`, `notool_shapley.csv` | `analyze()` на пяти промптах, где не нужен ни один инструмент. Пять — синтетическое малое число, для более сильных результатов, очевидно, нужно больше. |
| `noise_floor_notool.json` | $\nu$ на каждом из этих пяти промптов отдельно |

Генерация стохастична (`temperature=0.5` зашита в библиотеке), поэтому
повторный прогон даст другие числа. Выводы ноутбука `02_agent_attribution.ipynb` от этого не зависят.

Прогон занимает несколько минут.

In [1]:
import sys; sys.path.insert(0, "..")
from pathlib import Path

import pandas as pd
import requests

from agent import MODEL_NAME, OLLAMA_URL, build_agent_shap, measure_noise_floor

OUT = Path("../data/agentshap")
OUT.mkdir(exist_ok=True)

models = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
assert MODEL_NAME in models, f"В Ollama нет {MODEL_NAME}: ollama pull {MODEL_NAME}. Есть: {models}"

bundle = build_agent_shap()   # при первом запуске клонирует TokenSHAP рядом с репозиторием
agent_shap, tool_names = bundle.agent_shap, bundle.tool_names
print("модель:", MODEL_NAME, "  инструменты:", tool_names)

модель: qwen2.5:3b   инструменты: ['get_weather', 'get_stock_price', 'calculate', 'get_news']


## Пример `analyze()` на промпте, где нужны инструменты

In [2]:
PROMPT = "What's the weather in New York and how is AAPL stock doing today?"
def run(prompt):
    # analyze() плюс бейзлайн, который библиотека держит в атрибутах, а не в results_df
    res, sv = agent_shap.analyze(prompt=prompt, sampling_ratio=0.0)
    base = {"combination_key": "baseline", "tools_available": "+".join(tool_names),
            "response": agent_shap._baseline_response, "similarity": None,
            **{f"used_{k}": v for k, v in agent_shap._baseline_tool_usage.items()}}
    res = pd.concat([pd.DataFrame([base]), res.drop(columns="indexes")], ignore_index=True)
    return res.assign(prompt=prompt), {"prompt": prompt, **sv}

finance, finance_shap = run(PROMPT)
finance.to_csv(OUT / "finance.csv", index=False)
pd.DataFrame([finance_shap]).to_csv(OUT / "finance_shapley.csv", index=False)
print(finance.drop(columns=["response", "prompt"]).to_string(index=False))
print("\nshapley:", {k: round(v, 4) for k, v in finance_shap.items() if k != "prompt"})


Analyzing 4 tools:
  - get_weather
  - get_stock_price
  - calculate
  - get_news

1. Getting baseline response (all tools)...



2. Testing tool combinations...



Testing tool combinations:   0%|          | 0/4 [00:00<?, ?it/s]


Testing tool combinations:  25%|██▌       | 1/4 [00:13<00:40, 13.58s/it]


Testing tool combinations:  50%|█████     | 2/4 [00:22<00:22, 11.05s/it]


Testing tool combinations:  75%|███████▌  | 3/4 [00:30<00:09,  9.33s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:36<00:00,  8.35s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:36<00:00,  9.25s/it]


3. Calculating similarities...

4. Computing Shapley values...

 Analysis complete!
                               combination_key                                tools_available similarity  used_get_weather  used_get_stock_price  used_get_news
                                      baseline get_weather+get_stock_price+calculate+get_news       None               1.0                   1.0            NaN
   calculate+get_news+get_stock_price_idx2,3,4             calculate+get_news+get_stock_price   0.225646               NaN                   2.0            1.0
       calculate+get_news+get_weather_idx1,3,4                 calculate+get_news+get_weather   0.263532               1.0                   NaN            1.0
 get_news+get_stock_price+get_weather_idx1,2,4           get_news+get_stock_price+get_weather   0.624593               1.0                   1.0            NaN
calculate+get_stock_price+get_weather_idx1,2,3          calculate+get_stock_price+get_weather   0.683045           

## Noise floor

По постановке, AgentSHAP сравнивает ответ с урезанным набором инструментов с одним ответом при полном наборе: $v(S) = \cos(\text{ответ}(S), \text{ответ}(N))$, вклад $\propto 1 - v(N \setminus \{i\})$. 

Здесь формула полагает, что $v(N) = 1$.

Но обычкновенно мы не запускаем модели детерминированно (и в общем смысле детерминированности нет). Кроме того, как гиперпараметр `temperature=0.5` зашита в библиотеке, и два прогона с **одним и тем же** набором расходятся: $\nu = \mathbb{E}[\cos(\text{ответ}(N), \text{ответ}'(N))] < 1$.

Разницу расхождения $1 - \nu$ библиотека размазывает по всем инструментам. То есть в идеальном мире мы хотим 
- $\nu = \mathbb{E}[\cos(\text{ответ}(N), \text{ответ}'(N))] = 1$ — тогда $v(N) = 1$ выполнено. 

в идеальном получаем, что $v(N) = 1$  выполнено быть не обязано. При использовании фреймворка ν нужен, чтобы заменить недостижимую единицу, $\text{вклад}_i = \max(0, \nu - v(N \setminus \{i\}))$,
и чтобы задать порог шума $2\sigma$. 


**Алгоритм**, для каждого промпта:
1. $n$ прогонов агента с полным набором (получаем ответы ответы $r_1, \dots, r_n$).
2. TF-IDF, обученный на всех $n$ ответах разом, как в `analyze()`.
3. Косинус $r_1$ с каждым из $r_2, \dots, r_n$: $r_1$ — бейзлайн, как в `analyze()`.

$\nu$ и $\sigma$ — среднее и стандартное отклонение этих $n - 1$ чисел (по всем промптам списка). $\sigma$ — разброс одного сравнения, а не погрешность $\nu$: все сравнения идут против одного $r_1$.

Здесь 3 промпта по 10 прогонов с полным тулсетом. Первый ответ каждого промпта — бейзлайн, остальные девять сравниваются с ним той же метрикой, что в `analyze()`.

In [3]:
NOISE_PROMPTS = [
    PROMPT,
    "What's the latest AI news and what is 17 plus 25?",
    "How is TSLA doing, and what's the weather in Tokyo?",
]
nf = measure_noise_floor(agent_shap, NOISE_PROMPTS, n_runs=10)
nf.to_json(OUT / "noise_floor.json")
print(nf)

noise floor  nu = 0.6653 ± 0.1079   (27 сравнений, 3 промпта, qwen2.5:3b)
смещение   1 - nu = 0.3347
поведенческий шум: 0.0% прогонов с другим набором инструментов


## Промпты, где не нужен ни один инструмент

In [4]:
NO_TOOL_PROMPTS = [
    "What is the capital of France?",
    "Who wrote Anna Karenina?",
    "Translate 'good morning' into Spanish.",
    "What does the acronym CPU stand for?",
    "Name three primary colors.",
]
coal, shap = [], []
for p in NO_TOOL_PROMPTS:
    res, sv = run(p)
    coal.append(res)
    shap.append(sv)

notool = pd.concat(coal, ignore_index=True)
notool.to_csv(OUT / "notool_coalitions.csv", index=False)
pd.DataFrame(shap).to_csv(OUT / "notool_shapley.csv", index=False)
print(pd.DataFrame(shap).set_index("prompt").round(4).to_string())
used = [c for c in notool.columns if c.startswith("used_")]
print("\nвызовы инструментов на этих промптах, по всем коалициям:")
print(notool.groupby("prompt", sort=False)[used].sum().to_string())


Analyzing 4 tools:
  - get_weather
  - get_stock_price
  - calculate
  - get_news

1. Getting baseline response (all tools)...



2. Testing tool combinations...



Testing tool combinations:   0%|          | 0/4 [00:00<?, ?it/s]


Testing tool combinations:  25%|██▌       | 1/4 [00:04<00:12,  4.19s/it]


Testing tool combinations:  50%|█████     | 2/4 [00:06<00:06,  3.35s/it]

...


Testing tool combinations:  50%|█████     | 2/4 [00:17<00:19,  9.69s/it]


Testing tool combinations:  75%|███████▌  | 3/4 [00:22<00:07,  7.42s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:28<00:00,  7.11s/it]


Testing tool combinations: 100%|██████████| 4/4 [00:28<00:00,  7.17s/it]


3. Calculating similarities...

4. Computing Shapley values...

 Analysis complete!
                                        get_weather  get_stock_price  calculate  get_news
prompt                                                                                   
What is the capital of France?               0.0000           0.3381     0.3305    0.3314
Who wrote Anna Karenina?                     0.4989           0.0931     0.4080    0.0000
Translate 'good morning' into Spanish.       0.3993           0.2683     0.0000    0.3324
What does the acronym CPU stand for?         0.2034           0.4602     0.0000    0.3364
Name three primary colors.                   0.3012           0.4065     0.0000    0.2923

вызовы инструментов на этих промптах, по всем коалициям:
                                        used_get_news
prompt                                               
What is the capital of France?                    0.0
Who wrote Anna Karenina?                          0.0
Translate '

## Noise floor на каждом промпте без инструментов

Потолок близости зависит от промпта. Тут ответ с данными инструмента почти детерминирован, свободный же текст — нет. Поэтому для проверки dummy всегда измеряем $\nu$ на тех же промптах, на которых считаем вклады.

In [5]:
import json
from dataclasses import asdict

nf_notool = {p: measure_noise_floor(agent_shap, [p], n_runs=10) for p in NO_TOOL_PROMPTS}
(OUT / "noise_floor_notool.json").write_text(
    json.dumps({p: asdict(nf) for p, nf in nf_notool.items()}, ensure_ascii=False, indent=2), encoding="utf-8")
for p, nf in nf_notool.items():
    print(f"{p:<40} nu = {nf.nu:.3f} ± {nf.sd:.3f}")

What is the capital of France?           nu = 0.316 ± 0.079
Who wrote Anna Karenina?                 nu = 0.318 ± 0.102
Translate 'good morning' into Spanish.   nu = 0.308 ± 0.104
What does the acronym CPU stand for?     nu = 0.424 ± 0.141
Name three primary colors.               nu = 0.274 ± 0.048
